# 09. Modelos de Deep Learning para detección de phishing

Este notebook entrena, compara y exporta modelos neuronales para clasificación binaria de URLs utilizando las mismas **25 características Top Global** del sistema LinkShield AI.

Objetivos:

1. Mantener una separación reproducible entre entrenamiento, validación y prueba.
2. Comparar tres arquitecturas de redes densas para datos tabulares.
3. Evaluar accuracy, precision, recall, F1-score, ROC-AUC y matriz de confusión.
4. Exportar todos los modelos y registrar el mejor candidato.
5. Comparar el mejor modelo neuronal con el Random Forest actualmente recomendado.

> El conjunto de prueba se utiliza solamente para la evaluación final. La selección durante el entrenamiento se realiza con validación.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import hashlib
import json
import random
import shutil
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers, regularizers

DATASET_PATH = Path("dataset-phishing-final-top25.csv")
OUTPUT_DIR = Path("models/deep_learning")
TARGET_COLUMN = "status"
RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE_FROM_TRAIN = 0.20
BATCH_SIZE = 64
MAX_EPOCHS = 80
THRESHOLD = 0.50

RANDOM_FOREST_REFERENCE = {
    "id": "random_forest",
    "display_name": "Random Forest Top 25",
    "accuracy": 0.9488188976377953,
    "precision": 0.9430051813471503,
    "recall": 0.9553805774278216,
    "f1": 0.9491525423728814,
    "roc_auc": 0.986329738091575,
}

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("scikit-learn:", sklearn.__version__)

TensorFlow: 2.21.0
Keras: 3.14.0
scikit-learn: 1.7.2


## 1. Reproducibilidad y hardware

Se fijan las semillas de Python, NumPy y TensorFlow. En Windows nativo, TensorFlow moderno utiliza CPU; esto no afecta la validez de las métricas, solamente el tiempo de entrenamiento.

In [2]:
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

physical_gpus = tf.config.list_physical_devices("GPU")
print("GPU disponibles:", physical_gpus if physical_gpus else "Ninguna; entrenamiento en CPU")

GPU disponibles: Ninguna; entrenamiento en CPU


## 2. Carga y validación del dataset

El archivo contiene 25 variables predictoras ya preparadas por el pipeline anterior y una columna objetivo `status`, donde `0` representa una URL legítima y `1` phishing.

In [3]:
dataframe = pd.read_csv(DATASET_PATH)
assert TARGET_COLUMN in dataframe.columns, "No se encontró la variable objetivo"
assert not dataframe.isna().any().any(), "El dataset contiene valores faltantes"

feature_names = [column for column in dataframe.columns if column != TARGET_COLUMN]
assert len(feature_names) == 25, f"Se esperaban 25 variables y se encontraron {len(feature_names)}"

X = dataframe[feature_names].astype("float32")
y = dataframe[TARGET_COLUMN].astype("int32")
assert set(y.unique()) == {0, 1}, "La variable objetivo debe ser binaria"

display(dataframe.head())
print("Dimensiones:", dataframe.shape)
print("Variables:", len(feature_names))
display(y.value_counts().rename(index={0: "Legítima", 1: "Phishing"}).to_frame("muestras"))

Dimensiones: (11429, 26)
Variables: 25


,google_index,domain_age,nb_hyperlinks,nb_www,phish_hints,safe_anchor,longest_word_path,ratio_digits_url,ratio_intHyperlinks,longest_words_raw,domain_in_title,length_words_raw,nb_dots,domain_registration_length,ratio_extRedirection,nb_hyphens,char_repeat,length_hostname,nb_slash,ratio_extHyperlinks,links_in_tags,avg_word_path,domain_with_copyright,ip,ratio_extMedia,status
0,0,-1.307508,-0.420945,1,-0.388992,-0.948603,-0.197675,-0.594606,-0.194131,-0.198991,1,-0.400668,0.379217,-0.549305,2.687599,-0.478010,0.224865,-0.193917,-0.685076,0.606135,0.674784,-0.082904,0,0,-0.605252,0
1,0,0.548497,-0.342987,0,-0.388992,1.610694,0.928947,1.875996,0.967376,0.751942,0,-0.400668,-1.081050,-0.510030,-0.596479,-0.478010,0.224865,0.177241,0.377519,-0.760630,1.156476,1.339596,1,1,-0.084187,1
2,0,-0.018794,-0.498903,0,-0.388992,1.610694,0.278973,1.092832,1.055922,0.072704,0,1.034997,1.109351,-0.587353,-0.596479,0.001132,-0.194519,2.682560,0.377519,-0.864823,1.156476,0.426797,1,1,-0.605252,1
3,1,-1.307508,0.370626,0,-0.388992,0.650958,-0.457664,-0.594606,0.984610,-0.470686,0,-0.939042,-0.350916,-0.528440,0.341829,-0.478010,-0.613902,-0.936234,-1.216374,-0.780909,1.156476,-0.712535,1,0,-0.512205,0
4,1,1.323334,0.088778,1,-0.388992,-0.948603,0.018983,-0.594606,-0.350388,-0.198991,1,-0.041752,-0.350916,-0.329610,1.419146,0.480274,0.015173,-0.565075,0.377519,0.790005,0.589780,0.266891,0,0,2.000075,0


,muestras
status,
Legítima,5715
Phishing,5714


## 3. División entrenamiento, validación y prueba

- **64 % entrenamiento:** actualización de pesos.
- **16 % validación:** early stopping y reducción de learning rate.
- **20 % prueba:** comparación final no utilizada durante el ajuste.

Todas las divisiones son estratificadas para conservar el equilibrio de clases.

In [4]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=VALIDATION_SIZE_FROM_TRAIN,
    random_state=RANDOM_STATE,
    stratify=y_train_full,
)

split_summary = pd.DataFrame({
    "partición": ["Entrenamiento", "Validación", "Prueba"],
    "muestras": [len(X_train), len(X_val), len(X_test)],
    "legítimas": [(y_train == 0).sum(), (y_val == 0).sum(), (y_test == 0).sum()],
    "phishing": [(y_train == 1).sum(), (y_val == 1).sum(), (y_test == 1).sum()],
})
display(split_summary)

,partición,muestras,legítimas,phishing
0,Entrenamiento,7314,3657,3657
1,Validación,1829,915,914
2,Prueba,2286,1143,1143


## 4. Arquitecturas candidatas

Se comparan tres redes diseñadas para datos tabulares:

1. **MLP compacta:** referencia neuronal de baja complejidad.
2. **MLP regularizada:** mayor capacidad, normalización por lotes, dropout y regularización L2.
3. **Red residual densa:** bloque residual para facilitar la propagación de gradientes.

Todas producen una probabilidad de phishing mediante una salida sigmoide.

In [5]:
def compile_binary_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="roc_auc"),
        ],
    )
    return model


def build_compact_mlp(input_dim):
    inputs = keras.Input(shape=(input_dim,), name="top25_features")
    x = layers.Dense(64, activation="relu")(inputs)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.10)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="phishing_probability")(x)
    return compile_binary_model(keras.Model(inputs, outputs, name="mlp_compact"))


def build_regularized_mlp(input_dim):
    inputs = keras.Input(shape=(input_dim,), name="top25_features")
    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="phishing_probability")(x)
    return compile_binary_model(keras.Model(inputs, outputs, name="mlp_regularized"))


def build_residual_mlp(input_dim):
    inputs = keras.Input(shape=(input_dim,), name="top25_features")
    x = layers.Dense(64)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    shortcut = x
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([shortcut, x])
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="phishing_probability")(x)
    return compile_binary_model(keras.Model(inputs, outputs, name="mlp_residual"))


model_builders = {
    "mlp_compact": {
        "display_name": "MLP Compacta",
        "description": "Red densa ligera con dos capas ocultas.",
        "builder": build_compact_mlp,
    },
    "mlp_regularized": {
        "display_name": "MLP Regularizada",
        "description": "Red profunda con Batch Normalization, Dropout y regularización L2.",
        "builder": build_regularized_mlp,
    },
    "mlp_residual": {
        "display_name": "MLP Residual",
        "description": "Red densa con conexión residual y normalización por lotes.",
        "builder": build_residual_mlp,
    },
}

for model_id, config in model_builders.items():
    preview = config["builder"](len(feature_names))
    print(model_id, "parámetros:", preview.count_params())
    del preview
keras.backend.clear_session()

mlp_compact parámetros: 3777
mlp_regularized parámetros: 14465
mlp_residual parámetros: 8449


## 5. Entrenamiento

Cada candidato utiliza:

- `EarlyStopping` sobre `val_loss` para recuperar los mejores pesos.
- `ReduceLROnPlateau` para disminuir el learning rate cuando la validación se estanca.
- Hasta 80 épocas, aunque el entrenamiento puede finalizar antes.

In [6]:
models = {}
histories = {}
training_rows = []

for index, (model_id, config) in enumerate(model_builders.items()):
    tf.keras.utils.set_random_seed(RANDOM_STATE + index)
    model = config["builder"](len(feature_names))
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            min_delta=1e-4,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=4,
            min_lr=1e-5,
            verbose=1,
        ),
    ]
    print("\n" + "=" * 72)
    print("ENTRENANDO:", config["display_name"])
    print("=" * 72)
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )
    models[model_id] = model
    histories[model_id] = history.history
    training_rows.append({
        "model_id": model_id,
        "epochs_completed": len(history.history["loss"]),
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
        "best_val_loss": float(np.min(history.history["val_loss"])),
    })

display(pd.DataFrame(training_rows))


ENTRENANDO: MLP Compacta
Epoch 1/80
115/115 - 1s - 11ms/step - accuracy: 0.8194 - loss: 0.4174 - precision: 0.8145 - recall: 0.8272 - roc_auc: 0.9054 - val_accuracy: 0.9147 - val_loss: 0.2289 - val_precision: 0.9220 - val_recall: 0.9059 - val_roc_auc: 0.9681 - learning_rate: 0.0010
Epoch 2/80
115/115 - 0s - 2ms/step - accuracy: 0.9174 - loss: 0.2156 - precision: 0.9206 - recall: 0.9136 - roc_auc: 0.9716 - val_accuracy: 0.9224 - val_loss: 0.1899 - val_precision: 0.9289 - val_recall: 0.9147 - val_roc_auc: 0.9771 - learning_rate: 0.0010
Epoch 3/80
115/115 - 0s - 3ms/step - accuracy: 0.9245 - loss: 0.1931 - precision: 0.9262 - recall: 0.9226 - roc_auc: 0.9770 - val_accuracy: 0.9300 - val_loss: 0.1800 - val_precision: 0.9319 - val_recall: 0.9278 - val_roc_auc: 0.9794 - learning_rate: 0.0010
Epoch 4/80
115/115 - 0s - 2ms/step - accuracy: 0.9300 - loss: 0.1868 - precision: 0.9300 - recall: 0.9300 - roc_auc: 0.9781 - val_accuracy: 0.9344 - val_loss: 0.1739 - val_precision: 0.9431 - val_recall

,model_id,epochs_completed,best_epoch,best_val_loss
0,mlp_compact,64,54,0.137640
1,mlp_regularized,67,57,0.147247
2,mlp_residual,40,30,0.143127


## 6. Curvas de aprendizaje

In [7]:
fig, axes = plt.subplots(len(histories), 2, figsize=(13, 4 * len(histories)))
for row, (model_id, history) in enumerate(histories.items()):
    display_name = model_builders[model_id]["display_name"]
    axes[row, 0].plot(history["loss"], label="Entrenamiento")
    axes[row, 0].plot(history["val_loss"], label="Validación")
    axes[row, 0].set_title(f"Loss - {display_name}")
    axes[row, 0].set_xlabel("Época")
    axes[row, 0].legend()
    axes[row, 1].plot(history["accuracy"], label="Entrenamiento")
    axes[row, 1].plot(history["val_accuracy"], label="Validación")
    axes[row, 1].set_title(f"Accuracy - {display_name}")
    axes[row, 1].set_xlabel("Época")
    axes[row, 1].legend()
plt.tight_layout()
plt.show()

<ipython-input-1-154e31ae682d>:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Evaluación final

La selección del mejor modelo neuronal se realiza por **F1-score**, porque equilibra precision y recall en la detección de phishing.

In [8]:
evaluation_rows = []
confusion_matrices = {}

for model_id, model in models.items():
    probabilities = model.predict(X_test, batch_size=256, verbose=0).reshape(-1)
    predictions = (probabilities >= THRESHOLD).astype("int32")
    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()
    confusion_matrices[model_id] = np.array([[tn, fp], [fn, tp]])
    row = {
        "model_id": model_id,
        "modelo": model_builders[model_id]["display_name"],
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions),
        "recall": recall_score(y_test, predictions),
        "f1": f1_score(y_test, predictions),
        "roc_auc": roc_auc_score(y_test, probabilities),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }
    evaluation_rows.append(row)
    print("\n", model_builders[model_id]["display_name"])
    print(classification_report(y_test, predictions, target_names=["Legítima", "Phishing"]))

results_df = pd.DataFrame(evaluation_rows).sort_values("f1", ascending=False).reset_index(drop=True)
metric_columns = ["accuracy", "precision", "recall", "f1", "roc_auc"]
display(results_df.assign(**{column: results_df[column].round(4) for column in metric_columns}))


 MLP Compacta
              precision    recall  f1-score   support

    Legítima       0.95      0.93      0.94      1143
    Phishing       0.93      0.95      0.94      1143

    accuracy                           0.94      2286
   macro avg       0.94      0.94      0.94      2286
weighted avg       0.94      0.94      0.94      2286


 MLP Regularizada
              precision    recall  f1-score   support

    Legítima       0.93      0.95      0.94      1143
    Phishing       0.95      0.93      0.94      1143

    accuracy                           0.94      2286
   macro avg       0.94      0.94      0.94      2286
weighted avg       0.94      0.94      0.94      2286


 MLP Residual
              precision    recall  f1-score   support

    Legítima       0.93      0.94      0.94      1143
    Phishing       0.94      0.93      0.93      1143

    accuracy                           0.94      2286
   macro avg       0.94      0.94      0.94      2286
weighted avg       0.94  

,model_id,modelo,accuracy,precision,recall,f1,roc_auc,tn,fp,fn,tp
0,mlp_regularized,MLP Regularizada,0.9418,0.9501,0.9326,0.9413,0.9826,1087,56,77,1066
1,mlp_compact,MLP Compacta,0.9401,0.9344,0.9466,0.9405,0.9820,1067,76,61,1082
2,mlp_residual,MLP Residual,0.9353,0.9430,0.9265,0.9347,0.9806,1079,64,84,1059


## 8. Matrices de confusión

In [9]:
fig, axes = plt.subplots(1, len(confusion_matrices), figsize=(16, 4.5))
for axis, (model_id, matrix) in zip(axes, confusion_matrices.items()):
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Legítima", "Phishing"],
        yticklabels=["Legítima", "Phishing"],
        ax=axis,
    )
    axis.set_title(model_builders[model_id]["display_name"])
    axis.set_xlabel("Predicción")
    axis.set_ylabel("Valor real")
plt.tight_layout()
plt.show()

<ipython-input-1-1981d1cbec9a>:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Comparación con el modelo recomendado actual

El mejor candidato neuronal se compara con el Random Forest Top 25. El modelo recomendado para la aplicación solamente cambia si supera objetivamente su F1-score.

In [10]:
best_row = results_df.iloc[0]
best_model_id = best_row["model_id"]
best_dl_f1 = float(best_row["f1"])
rf_f1 = RANDOM_FOREST_REFERENCE["f1"]
recommended_for_application = (
    best_model_id if best_dl_f1 > rf_f1 else RANDOM_FOREST_REFERENCE["id"]
)

comparison = pd.DataFrame([
    {
        "modelo": best_row["modelo"],
        "tipo": "Deep Learning",
        "accuracy": best_row["accuracy"],
        "precision": best_row["precision"],
        "recall": best_row["recall"],
        "f1": best_row["f1"],
        "roc_auc": best_row["roc_auc"],
    },
    {
        "modelo": RANDOM_FOREST_REFERENCE["display_name"],
        "tipo": "Machine Learning",
        **{key: RANDOM_FOREST_REFERENCE[key] for key in ("accuracy", "precision", "recall", "f1", "roc_auc")},
    },
]).sort_values("f1", ascending=False)
display(comparison.assign(**{
    column: comparison[column].round(4)
    for column in ("accuracy", "precision", "recall", "f1", "roc_auc")
}))
print("Mejor modelo Deep Learning:", model_builders[best_model_id]["display_name"])
print("Modelo recomendado para la aplicación:", recommended_for_application)

Mejor modelo Deep Learning: MLP Regularizada
Modelo recomendado para la aplicación: random_forest


,modelo,tipo,accuracy,precision,recall,f1,roc_auc
1,Random Forest Top 25,Machine Learning,0.9488,0.9430,0.9554,0.9492,0.9863
0,MLP Regularizada,Deep Learning,0.9418,0.9501,0.9326,0.9413,0.9826


## 10. Exportación de modelos y metadatos

Se exportan:

- Los tres modelos `.keras`.
- Una copia del ganador como `mejor_modelo_deep_learning.keras`.
- Métricas CSV y registro JSON.
- Orden de características y scaler compartido.
- Muestra de referencia para explicabilidad SHAP futura.

In [11]:
def artifact_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()[:12]


exported_at = datetime.now(timezone.utc).isoformat()
artifacts = []
for _, result in results_df.iterrows():
    model_id = result["model_id"]
    artifact_path = OUTPUT_DIR / f"{model_id}.keras"
    models[model_id].save(artifact_path)
    artifacts.append({
        "id": model_id,
        "display_name": model_builders[model_id]["display_name"],
        "description": model_builders[model_id]["description"],
        "technical_name": models[model_id].name,
        "artifact": artifact_path.name,
        "artifact_id": artifact_hash(artifact_path),
        "recommended_deep_learning": model_id == best_model_id,
        "recommended_for_application": model_id == recommended_for_application,
        "threshold": THRESHOLD,
        "parameter_count": int(models[model_id].count_params()),
        "metrics": {key: float(result[key]) for key in ("accuracy", "precision", "recall", "f1", "roc_auc")},
        "confusion_matrix": {key: int(result[key]) for key in ("tn", "fp", "fn", "tp")},
        "training": next(row for row in training_rows if row["model_id"] == model_id),
    })

best_artifact = OUTPUT_DIR / f"{best_model_id}.keras"
shutil.copy2(best_artifact, OUTPUT_DIR / "mejor_modelo_deep_learning.keras")
joblib.dump(feature_names, OUTPUT_DIR / "features_deep_learning_top25.pkl")
joblib.dump(X_train.sample(n=50, random_state=RANDOM_STATE), OUTPUT_DIR / "shap_background_top25.pkl")
source_scaler = Path("scaler_top.pkl")
if source_scaler.exists():
    shutil.copy2(source_scaler, OUTPUT_DIR / "scaler_top.pkl")

results_df.to_csv(OUTPUT_DIR / "metricas_deep_learning_top25.csv", index=False)
for model_id, history in histories.items():
    (OUTPUT_DIR / f"history_{model_id}.json").write_text(
        json.dumps({key: [float(value) for value in values] for key, values in history.items()}, indent=2),
        encoding="utf-8",
    )

registry = {
    "schema_version": 1,
    "exported_at": exported_at,
    "framework": "TensorFlow/Keras",
    "tensorflow_version": tf.__version__,
    "keras_version": keras.__version__,
    "dataset": {
        "name": DATASET_PATH.name,
        "total_rows": len(dataframe),
        "train_rows": len(X_train),
        "validation_rows": len(X_val),
        "test_rows": len(X_test),
        "test_legitimate_rows": int((y_test == 0).sum()),
        "test_phishing_rows": int((y_test == 1).sum()),
    },
    "features": feature_names,
    "best_deep_learning_model_id": best_model_id,
    "application_recommended_model_id": recommended_for_application,
    "random_forest_reference": RANDOM_FOREST_REFERENCE,
    "models": artifacts,
}
(OUTPUT_DIR / "deep_learning_registry.json").write_text(
    json.dumps(registry, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(OUTPUT_DIR / "metadata_mejor_modelo_deep_learning.json").write_text(
    json.dumps(next(item for item in artifacts if item["id"] == best_model_id), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Artefactos exportados en:", OUTPUT_DIR.resolve())
print("Ganador Deep Learning:", best_model_id)
print("Recomendado para aplicación:", recommended_for_application)

Artefactos exportados en: C:\phishing-url-classifier\phishing-url-ai\models\deep_learning
Ganador Deep Learning: mlp_regularized
Recomendado para aplicación: random_forest


## 11. Verificación de artefactos exportados

In [12]:
loaded_model = keras.models.load_model(OUTPUT_DIR / "mejor_modelo_deep_learning.keras")
loaded_features = joblib.load(OUTPUT_DIR / "features_deep_learning_top25.pkl")
assert loaded_features == feature_names
sample_probability = float(loaded_model.predict(X_test.iloc[[0]], verbose=0)[0, 0])
original_probability = float(models[best_model_id].predict(X_test.iloc[[0]], verbose=0)[0, 0])
assert abs(sample_probability - original_probability) < 1e-6
print("Modelo recargado correctamente")
print("Entrada:", loaded_model.input_shape, "Salida:", loaded_model.output_shape)
print("Probabilidad de prueba:", round(sample_probability, 6))

Modelo recargado correctamente
Entrada: (None, 25) Salida: (None, 1)
Probabilidad de prueba: 0.996894


## Conclusión

El registro exportado permite integrar posteriormente estos modelos en la API y en el selector de la aplicación. La decisión recomendada queda fundamentada en el F1-score del mismo conjunto de prueba usado para comparar los modelos clásicos.

Para producción deben conservarse juntos el modelo, el orden de características, el scaler y los metadatos. La probabilidad neuronal deberá acompañarse de explicabilidad compatible antes de habilitar el modelo en Anita y en los historiales.